# 01 · Dataset explorer

Reads the recordings on Drive and answers three questions before a single epoch is trained:

1. **How much data is there?** counts, per-class distribution, durations.
2. **Is any of it broken?** corrupted, empty, silent, stereo, wrong sample rate, duplicated.
3. **Is it balanced enough to train on?** thin classes are flagged.

Nothing is modified here — this notebook only reads. Cleaning happens in `02_preprocessing`.

Expected layout on Drive:

```
MyDrive/Dhikr Speech Dataset/
├── dataset/001/*.wav        one folder per phrase id
├── dataset/unknown/*.wav    filler / out-of-vocabulary audio
└── phrases.json             [{"id": 1, "text": "سبحان الله"}, ...]
```


In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


## 1 · Locate the dataset

Paths come from `configs/config.yaml`. Change `paths.drive_root` / `paths.project_dir` there if
your dataset lives somewhere else — never edit paths in the notebooks.

In [ ]:
from pathlib import Path

paths = config.paths
rows = [
    ("dataset", paths.dataset_path),
    ("phrases.json", paths.phrases_path),
    ("processed", paths.processed_path),
    ("checkpoints", paths.checkpoints_path),
    ("exports", paths.exports_path),
    ("logs", paths.logs_path),
    ("reports", paths.reports_path),
    ("noise (optional)", paths.noise_path),
]
for name, path in rows:
    print("%-18s %-6s %s" % (name, "ok" if Path(path).exists() else "MISSING", path))

if not paths.dataset_path.is_dir():
    raise FileNotFoundError(
        "dataset folder not found: %s\n"
        "Upload your recordings there, or point paths.drive_root / paths.project_dir "
        "at the right place in configs/config.yaml." % paths.dataset_path
    )


## 2 · Phrases

`phrases.json` maps a class id to the Arabic phrase. Folder `001` is phrase id `1`.

In [ ]:
import pandas as pd

from src.dataset import load_phrases

phrases = load_phrases(paths.phrases_path)
phrase_table = pd.DataFrame(
    [
        {
            "id": phrase.id,
            "folder": phrase.folder,
            "text": phrase.text,
            "folder exists": (paths.dataset_path / phrase.folder).is_dir(),
            "recordings": len(list((paths.dataset_path / phrase.folder).glob("*")))
            if (paths.dataset_path / phrase.folder).is_dir()
            else 0,
        }
        for phrase in phrases
    ]
)
print("%d phrases declared" % len(phrases))
phrase_table


## 3 · Index every recording

Folders are the class vocabulary: numeric folders are phrases, `unknown` is the filler class that
teaches the model to stay quiet on everything else.

In [ ]:
from src.dataset import scan_dataset

index = scan_dataset(
    paths.dataset_path,
    phrases,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
)

counts = index.counts()
count_table = pd.DataFrame(
    [
        {
            "class": label,
            "recordings": count,
            "share": count / max(len(index), 1),
            "text": index.label_text(label),
        }
        for label, count in counts.items()
    ]
).sort_values("recordings", ascending=False)

print("recordings :", len(index))
print("classes    :", index.num_classes)
count_table


## 4 · Validate

Every file is opened and decoded. This is the slow cell — a few minutes for thousands of clips —
and it is what finds silent takes and duplicated uploads.

Set `DEEP = False` to only read file headers (fast, but skips silence and duplicate detection).

In [ ]:
from src.dataset import validate_dataset

DEEP = True

def progress(done: int, total: int, every: int = 200) -> None:
    """Print progress without flooding the notebook output."""
    if done == total or done % every == 0:
        print("  %d / %d" % (done, total), flush=True)

report = validate_dataset(index, config.audio, deep=DEEP, progress=progress)
print()
print(report.summary())


### Issues found

| kind | meaning | what to do |
|---|---|---|
| `corrupted` | the file cannot be decoded | delete or re-record |
| `empty` | zero-length file | delete |
| `silent` | RMS below `audio.silence_dbfs` | delete — it teaches the model nothing |
| `stereo` | more than one channel | harmless, notebook 02 downmixes it |
| `sample_rate` | not 16 kHz | harmless, notebook 02 resamples it |
| `too_short` / `too_long` | outside `audio.min_duration` / `max_duration` | review; usually a truncated take |
| `duplicate` | identical audio to another file | delete the copy — duplicates leak across the train/test split |

`corrupted`, `empty` and `silent` files are excluded from preprocessing automatically.

In [ ]:
issues = report.issues_dataframe()
if len(issues):
    display(issues.groupby("kind").size().rename("count").to_frame())
    display(issues.head(50))
else:
    print("no issues found")

unusable = report.unusable_paths()
print()
print("%d file(s) will be excluded from preprocessing" % len(unusable))
for path in unusable[:20]:
    print("  ", path)


## 5 · Statistics and distribution

In [ ]:
from src import visualization as viz

stats = report.stats
print("total files    :", stats.total_files)
print("total classes  :", stats.total_classes)
print("total audio    : %.1f minutes" % (stats.total_duration / 60.0))
print("duration (s)   : mean %.2f | median %.2f | min %.2f | max %.2f" % (
    stats.mean_duration, stats.median_duration, stats.min_duration, stats.max_duration))

MIN_PER_CLASS = 50  # rule of thumb for a usable keyword-spotting class

figure = viz.plot_class_distribution(counts, highlight_below=MIN_PER_CLASS)
viz.save_figure(figure, paths.reports_path / "01_class_distribution.png")

durations = [item.duration for item in report.file_stats if item.ok and item.duration > 0]
figure = viz.plot_duration_histogram(
    durations,
    min_duration=config.audio.min_duration,
    max_duration=config.audio.max_duration,
)
viz.save_figure(figure, paths.reports_path / "01_duration_histogram.png")

thin = [label for label, count in counts.items() if count < MIN_PER_CLASS]
if thin:
    print()
    print("classes with fewer than %d recordings:" % MIN_PER_CLASS, ", ".join(thin))


## 6 · Listen to one sample per class

A quick ear check catches problems no validator can: the wrong phrase in a folder, a clipped
microphone, background speech.

In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio
from src.features import LogMelExtractor

rng = np.random.default_rng(config.seed)
extractor = LogMelExtractor(config.features, config.audio.sample_rate)
by_class = index.by_class()

previews, titles = [], []
for label in index.class_names:
    samples_for_class = by_class.get(label, [])
    if not samples_for_class:
        continue
    chosen = samples_for_class[int(rng.integers(len(samples_for_class)))]
    try:
        clip = load_audio(chosen.path, config.audio.sample_rate)
    except Exception as error:
        print("could not load", chosen.path, error)
        continue
    print("%-10s %s" % (label, chosen.path.name))
    display(Audio(clip, rate=config.audio.sample_rate))
    previews.append(extractor(clip))
    titles.append(label)

if previews:
    figure = viz.plot_feature_grid(previews, titles, hop_ms=config.features.hop_ms)
    viz.save_figure(figure, paths.reports_path / "01_class_previews.png")


## 7 · Save the validation report

Written to `reports/` on Drive:

* `validation_report.json` — statistics, issue counts, every issue
* `validation_report_files.csv` — one row per file (duration, rate, channels, RMS, content hash)

Notebook 02 reads this file to know which recordings to skip.

In [ ]:
written = report.save(paths.reports_path)
for kind, path in written.items():
    print("%-6s %s" % (kind, path))

print()
if report.is_clean:
    print("dataset is clean — continue with 02_preprocessing.ipynb")
else:
    print("%d issue(s) recorded." % len(report.issues))
    print("Delete the duplicates and silent takes, then re-run this notebook.")
    print("Everything else is handled automatically by 02_preprocessing.ipynb.")
